# Data Cleaning — custom_esci_with_user_query.csv

This notebook performs data cleaning on the custom ESCI dataset:
- Load the dataset (CSV or Excel, auto-detected)
- Drop junk columns
- Fix invalid labels (misaligned rows)
- Strip whitespace
- Remove repeated filler text from queries/titles
- Remove duplicates
- Validate label distribution
- Save cleaned dataset as CSV for training

In [1]:
import pandas as pd
import os
import re
from collections import Counter
from IPython.display import display

## 1. Load the Dataset

In [2]:
DATASET_DIR = r"C:\Moi\Thesis\Code\RetailTalkFolder\classification_identification\custom_esci"
FILE_PATH = os.path.join(DATASET_DIR, "custom_esci_with_user_query.csv")

# Auto-detect file format
if FILE_PATH.endswith('.csv'):
    df = pd.read_csv(FILE_PATH, encoding='latin-1')
else:
    df = pd.read_excel(FILE_PATH, engine='openpyxl')

print(f"Original shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)

Original shape: (71618, 4)
Columns: ['query', 'product_title', 'label', 'user_query']


,query,product_title,label,user_query
0,summer dress,Boho Floral V-Neck Summer Dress for Women,E,Any good options for summer dress?
1,summer dress,Casual Sleeveless Sun Dress with Pockets,E,Any good options for summer dress?
2,summer dress,Lightweight Cotton Maxi Skirt,S,Any good options for summer dress?
3,summer dress,Wide Brim Straw Sun Hat,C,Any good options for summer dress?
4,summer dress,Organic Extra Virgin Olive Oil 500ml,I,Any good options for summer dress?
5,running shoes,Nike Air Zoom Pegasus Running Shoes,E,Whatâs a good running shoes for daily use?
6,running shoes,Adidas Ultraboost Lightweight Sneakers,S,Whatâs a good running shoes for daily use?
7,running shoes,Performance Moisture-Wicking Athletic Socks,C,Whatâs a good running shoes for daily use?
8,running shoes,Stainless Steel Non-Stick Frying Pan,I,Whatâs a good running shoes for daily use?
9,iphone charger,Apple 20W USB-C Power Adapter Wall Charger,E,Trying to find a reliable iphone charger


## 2. Drop Junk Columns

In [3]:
# Drop any 'Unnamed' columns (artifacts from Excel)
unnamed_cols = [col for col in df.columns if col.startswith('Unnamed')]
if unnamed_cols:
    print(f"Dropping junk columns: {unnamed_cols}")
    df = df.drop(columns=unnamed_cols)
else:
    print("No junk columns found.")

print(f"Shape after drop: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

No junk columns found.
Shape after drop: (71618, 4)
Columns: ['query', 'product_title', 'label', 'user_query']


## 3. Strip Whitespace

In [4]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

# Uppercase labels for consistency
df['label'] = df['label'].str.upper()

print("Whitespace stripped and labels uppercased.")

Whitespace stripped and labels uppercased.


## 3.5 Remove Repeated Filler Text from Queries

Some queries have repeated padding like `"Set Pack Set Pack Set Pack Set"` appended.
This step detects and removes those repeated trailing patterns.

In [5]:
def remove_repeated_filler(text):
    """
    Remove repetitive word patterns from text:
    1. Word-cycling: 'beef meat can food meal beef meat can meal...' → 'beef meat can food meal'
       Detected via low unique-word ratio; fixed by keeping first occurrence of each word.
    2. Trailing repeats: '... Set Pack Set Pack Set Pack Set' → strips the repeated tail.
    """
    if not isinstance(text, str):
        return text

    words = text.split()
    if len(words) < 6:
        return text

    # --- Method 1: Word-cycling (low unique word ratio) ---
    unique_ratio = len(set(w.lower() for w in words)) / len(words)
    if unique_ratio <= 0.6:
        seen = set()
        result = []
        for w in words:
            wl = w.lower()
            if wl not in seen:
                seen.add(wl)
                result.append(w)
        return ' '.join(result)

    # --- Method 2: Trailing repeated patterns ---
    for pattern_len in range(1, 5):
        if len(words) < pattern_len * 3:
            continue

        candidate = words[-pattern_len:]

        repeat_count = 0
        pos = len(words)
        while pos >= pattern_len:
            chunk = words[pos - pattern_len : pos]
            if chunk == candidate:
                repeat_count += 1
                pos -= pattern_len
            else:
                break

        if repeat_count >= 3:
            clean_words = words[:pos + pattern_len]
            return ' '.join(clean_words)

    return text

# Save originals before cleaning so we can count changes without re-reading the file
original_queries = df['query'].copy()

# Apply to both columns
df['query'] = df['query'].apply(remove_repeated_filler)
df['product_title'] = df['product_title'].apply(remove_repeated_filler)

# Count changed rows (fillna handles NaN safely)
query_changed = (original_queries.fillna('') != df['query'].fillna('')).sum()
print(f"Queries cleaned (filler removed): {query_changed} out of {len(df)}")
print("Filler removal also applied to product_title column.")

Queries cleaned (filler removed): 4393 out of 71618
Filler removal also applied to product_title column.


## 4. Check for Invalid Labels

In [6]:
print(f"Current shape after filler removal: {df.shape}")
df.head(3)

Current shape after filler removal: (71618, 4)


,query,product_title,label,user_query
0,summer dress,Boho Floral V-Neck Summer Dress for Women,E,Any good options for summer dress?
1,summer dress,Casual Sleeveless Sun Dress with Pockets,E,Any good options for summer dress?
2,summer dress,Lightweight Cotton Maxi Skirt,S,Any good options for summer dress?


In [7]:
valid_labels = {'E', 'S', 'C', 'I'}
invalid_mask = ~df['label'].isin(valid_labels)
invalid_count = invalid_mask.sum()

print(f"Invalid label rows: {invalid_count}")

if invalid_count > 0:
    # Some rows have misaligned columns: description landed in 'label', actual label in 'user_query'
    # Fix: where label is invalid but user_query IS a valid label, swap them
    fixable_mask = invalid_mask & df['user_query'].isin(valid_labels)
    fixable_count = fixable_mask.sum()
    print(f"  Fixable (label in user_query column): {fixable_count}")

    if fixable_count > 0:
        df.loc[fixable_mask, 'label'] = df.loc[fixable_mask, 'user_query']
        df.loc[fixable_mask, 'user_query'] = pd.NA
        print(f"  Realigned {fixable_count} rows.")

    # Re-check for remaining invalid labels (e.g. header rows, Excel artifacts)
    still_invalid = ~df['label'].isin(valid_labels)
    remaining = still_invalid.sum()
    if remaining > 0:
        print(f"\n  Remaining unfixable rows: {remaining}")
        display(df[still_invalid].head(10))
        df = df[~still_invalid].copy()
        print(f"  Dropped {remaining} unfixable rows.")

    print(f"\nShape after label fix: {df.shape}")
else:
    print("All labels are valid.")

Invalid label rows: 18333
  Fixable (label in user_query column): 0

  Remaining unfixable rows: 18333


,query,product_title,label,user_query
53285,query,product_title,NaN,label
53286,query,product_title,NaN,product_description
53287,lucky me chicken noodles,Lucky Me Chicken Noodles,NaN,Instant noodles with chicken flavor in a singl...
53288,lucky me chicken noodles,Nissin Cup Noodles Chicken,NaN,Instant cup noodles with chicken broth flavor
53289,lucky me chicken noodles,Datu Puti Soy Sauce,NaN,A savory soy sauce commonly used as noodle con...
53290,lucky me chicken noodles,Safeguard Soap,NaN,Antibacterial bath soap in a small bar
53291,century tuna in oil,Century Tuna in Oil,NaN,Canned tuna flakes preserved in vegetable oil
53292,century tuna in oil,555 Sardines in Tomato Sauce,NaN,Canned fish in a savory tomato sauce
53293,century tuna in oil,Skyflakes Crackers,NaN,Plain crackers often eaten with canned tuna
53294,century tuna in oil,Ariel Detergent Sachet,NaN,Concentrated laundry powder for washing clothes


  Dropped 18333 unfixable rows.

Shape after label fix: (53285, 4)


## 5. Check for Nulls

In [8]:
null_counts = df.isnull().sum()
print("Null counts per column:")
print(null_counts)

if null_counts.sum() > 0:
    print(f"\nDropping {null_counts.sum()} rows with nulls...")
    df = df.dropna(subset=['query', 'product_title', 'label'])
    print(f"Shape after dropping nulls: {df.shape}")
else:
    print("No nulls found.")

Null counts per column:
query            0
product_title    0
label            0
user_query       0
dtype: int64
No nulls found.


## 6. Remove Duplicates

In [9]:
num_duplicates = df.duplicated().sum()
print(f"Duplicate rows: {num_duplicates}")

if num_duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed {num_duplicates} duplicates.")
    print(f"Shape after dedup: {df.shape}")
else:
    print("No duplicates found.")

Duplicate rows: 0
No duplicates found.


## 7. Dataset Summary

In [10]:
print(f"Final shape: {df.shape}")
print(f"Unique queries: {df['query'].nunique()}")
print(f"Unique products: {df['product_title'].nunique()}")
print(f"\nLabel distribution:")

label_names = {'E': 'Exact', 'S': 'Substitute', 'C': 'Complement', 'I': 'Irrelevant'}
label_counts = df['label'].value_counts()
for label in ['E', 'S', 'C', 'I']:
    count = label_counts.get(label, 0)
    pct = count / len(df) * 100
    print(f"  {label} ({label_names[label]}): {count} ({pct:.1f}%)")

Final shape: (53285, 4)
Unique queries: 10445
Unique products: 14918

Label distribution:
  E (Exact): 12842 (24.1%)
  S (Substitute): 13175 (24.7%)
  C (Complement): 13293 (24.9%)
  I (Irrelevant): 13975 (26.2%)


In [11]:
# Show sample rows per label
for label in ['E', 'S', 'C', 'I']:
    print(f"\n--- {label} ({label_names[label]}) sample ---")
    display(df[df['label'] == label].sample(min(5, len(df[df['label'] == label])), random_state=42))


--- E (Exact) sample ---


,query,product_title,label,user_query
35386,white t-shirt for men,Bench Men Basic White Tee V-Neck,E,Looking ako ng white t-shirt for men na okay q...
48235,basin plastic,Plastic Basin Medium 1pc,E,Meron ba kayong basin plastic na budget friendly?
34293,school bag,Backpack for School Blue,E,Need recommendations for school bag
36977,kopiko,Kopiko Brown Coffee 3-in-1 Mix,E,Trying to find a reliable kopiko
36258,del monte,Del Monte Tomato Sauce 250g,E,Need recommendations for del monte



--- S (Substitute) sample ---


,query,product_title,label,user_query
9718,c2 pack set tea,Lipton Iced Tea Green Tea 500ml,S,Pwede ba makahanap ng c2 pack set tea na mura ...
9133,chicken cube set,Maggi Chicken Broth Cubes 60g,S,Any good options for chicken cube set?
16496,sunsilk strong and long,Palmolive Intensive Moisture Shampoo,S,Pwede ba makahanap ng sunsilk strong and long ...
33880,ponds facial foam,Master Oil Control Facial Scrub 100g,S,Need recommendations for ponds facial foam
3433,kethup,Heinz Tomato Ketchup 300g,S,Need recommendations for kethup



--- C (Complement) sample ---


,query,product_title,label,user_query
50451,walistambo,Dustpan Plastic,C,Need ko ng walistambo na matibay sana
13436,purefoods nuggets 200g,UFC Ketchup,C,Looking for an affordable purefoods nuggets 200g
25986,green alcohol cross ethyl 500,Cotton Balls 50pcs Pack,C,Meron ba kayong green alcohol cross ethyl 500 ...
51357,toy bow,Target Board for Toy Archery,C,Trying to find a reliable toy bow
27299,orange juice sachet,Plastic Pitcher 1 Liter,C,May alam ba kayong magandang orange juice sachet?



--- I (Irrelevant) sample ---


,query,product_title,label,user_query
10230,mug litro litro set,Anker Fast Charging Cable,I,Whatâs a good mug litro litro set for daily ...
32429,mr chips,Silver Swan Soy Sauce 1L,I,Pwede ba makahanap ng mr chips na mura lang?
25203,soy sauce silver swan,Sunsilk Shampoo,I,Any good options for soy sauce silver swan?
36932,creamsilk,Milo Chocolate Drink 22g,I,Need recommendations for creamsilk
14413,pampatuyo ng buhok,Magic Sarap Seasoning 8g,I,Pwede ba makahanap ng pampatuyo ng buhok na mu...


## 8. Save Cleaned Dataset

In [12]:
# Save cleaned CSV
CSV_OUTPUT = os.path.join(DATASET_DIR, "custom_esci_template.csv")
df.to_csv(CSV_OUTPUT, index=False)
print(f"Saved cleaned CSV to: {CSV_OUTPUT}")
print(f"Rows: {len(df)}")

Saved cleaned CSV to: C:\Moi\Thesis\Code\RetailTalkFolder\classification_identification\custom_esci\custom_esci_template.csv
Rows: 53285


In [13]:
# Verify saved CSV
df_verify = pd.read_csv(CSV_OUTPUT)
print(f"Verified CSV shape: {df_verify.shape}")
print(f"Remaining duplicates: {df_verify.duplicated().sum()}")
print(f"Invalid labels: {(~df_verify['label'].isin(valid_labels)).sum()}")

Verified CSV shape: (53285, 4)
Remaining duplicates: 0
Invalid labels: 0
